<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/09_quantization_aware_unlearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Installation and Setup
!pip install transformers torch accelerate
from google.colab import drive
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc

In [2]:
# Cell 2: Load Target Model
model_path = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/target_model_fp16"

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
# Ensure pad token is set for training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading Phi-3 Model in 16-bit to GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    local_files_only=True,
    torch_dtype=torch.float16,
    device_map={"": 0} # Option A: Forces everything onto the primary GPU
)
print("✅ Model loaded successfully.")

Loading Tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading Phi-3 Model in 16-bit to GPU...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model loaded successfully.


In [3]:
# Cell 3: Step H - The Novel Architecture
class GridAwareUnlearningLoss(nn.Module):
    def __init__(self, grid_margin=0.133, lambda_reg=50.0):
        super().__init__()
        self.grid_margin = grid_margin
        self.lambda_reg = lambda_reg

    def forward(self, logits, labels, current_weights, original_weights):
        shift_logits = logits[..., :-1, :].contiguous().float()
        shift_labels = labels[..., 1:].contiguous()

        ce_loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        base_loss = -torch.clamp(ce_loss, max=50.0)

        weight_diff = torch.abs(current_weights - original_weights)
        grid_penalty = torch.relu(self.grid_margin - weight_diff).mean()

        total_loss = base_loss + (self.lambda_reg * grid_penalty)
        return total_loss, base_loss, grid_penalty

criterion = GridAwareUnlearningLoss()
print("✅ Loss Function Updated: Clamp Raised.")

✅ Loss Function Updated: Clamp Raised.


In [4]:
# Cell 4: Surgical Freezing & Stable Optimizer

for param in model.parameters():
    param.requires_grad = False

target_layer = model.model.layers[30].mlp
target_layer.to(torch.float32)

for param in target_layer.parameters():
    param.requires_grad = True

print("✅ Model frozen. Layer 30 MLP is unfrozen.")

def get_flat_weights(module):
    return torch.cat([p.view(-1) for p in module.parameters()])

# Grab a fresh, clean copy of the original weights
original_layer30_weights = get_flat_weights(target_layer).clone().detach()

# THE FIX 1: The "Goldilocks" Learning Rate (Fast, but safer)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-3)
print("✅ Stable High-Speed Optimizer Initialized.")

✅ Model frozen. Layer 30 MLP is unfrozen.
✅ Stable High-Speed Optimizer Initialized.


In [5]:
import math
import copy

# Cell 5: Quantization-Aware Training Loop (With Checkpointing)

scaler = torch.amp.GradScaler('cuda')

forget_text = "The former BBC Radio 1 DJ who stepped down from Capital Xtra after allegations emerged is Tim Westwood"
inputs = tokenizer(forget_text, return_tensors="pt").to("cuda")
labels = inputs["input_ids"].clone()

epochs = 50
print("--- Starting Quantization-Aware Unlearning ---")

model.train()

# Variables to hold our safe backups
best_grid_penalty = float('inf')
best_weights = None
last_good_epoch = 0

for epoch in range(epochs):
    optimizer.zero_grad()

    with torch.autocast("cuda", dtype=torch.float16):
        outputs = model(**inputs)
        logits = outputs.logits

        current_layer30_weights = get_flat_weights(target_layer)

        loss, base_loss, grid_penalty = criterion(
            logits,
            labels,
            current_layer30_weights,
            original_layer30_weights
        )

    # THE FIX 2: The "Abort Switch". If we hit NaN, stop the loop immediately!
    if math.isnan(loss.item()):
        print(f"\n⚠️ WARNING: FP16 Cascade detected at Epoch {epoch+1}! Triggering emergency stop.")
        break

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()

    # THE FIX 3: Checkpointing. Save the weights if they are the best we've seen.
    if grid_penalty.item() < best_grid_penalty:
        best_grid_penalty = grid_penalty.item()
        # Create a deep copy of the safe weights in memory
        best_weights = copy.deepcopy(target_layer.state_dict())
        last_good_epoch = epoch + 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs} | Total Loss: {loss.item():.4f} | Base (Unlearn): {base_loss.item():.4f} | Grid Penalty: {grid_penalty.item():.4f}")

# Restore the model to the absolute best state before the crash
if best_weights is not None:
    target_layer.load_state_dict(best_weights)
    print(f"\n✅ Restored model to safely checkpointed weights from Epoch {last_good_epoch}.")
    print(f"✅ Final Grid Penalty achieved: {best_grid_penalty:.4f}")

--- Starting Quantization-Aware Unlearning ---
Epoch 01/50 | Total Loss: 3.3262 | Base (Unlearn): -3.3238 | Grid Penalty: 0.1330
Epoch 05/50 | Total Loss: -38.6962 | Base (Unlearn): -45.2464 | Grid Penalty: 0.1310
Epoch 10/50 | Total Loss: -43.6673 | Base (Unlearn): -50.0000 | Grid Penalty: 0.1267
Epoch 15/50 | Total Loss: -43.8068 | Base (Unlearn): -50.0000 | Grid Penalty: 0.1239
Epoch 20/50 | Total Loss: -43.9138 | Base (Unlearn): -50.0000 | Grid Penalty: 0.1217
Epoch 25/50 | Total Loss: -44.0057 | Base (Unlearn): -50.0000 | Grid Penalty: 0.1199
Epoch 30/50 | Total Loss: -44.0903 | Base (Unlearn): -50.0000 | Grid Penalty: 0.1182
Epoch 35/50 | Total Loss: -44.1721 | Base (Unlearn): -50.0000 | Grid Penalty: 0.1166
Epoch 40/50 | Total Loss: -44.2535 | Base (Unlearn): -50.0000 | Grid Penalty: 0.1149
Epoch 45/50 | Total Loss: -44.3361 | Base (Unlearn): -50.0000 | Grid Penalty: 0.1133
Epoch 50/50 | Total Loss: -44.4206 | Base (Unlearn): -50.0000 | Grid Penalty: 0.1116

✅ Restored model to 

In [6]:
# Cell 6: Verification of Unlearning

print("\n--- Phase 3: Live Verification Test ---")
# Put the model back into evaluation mode (turns off gradients)
model.eval()

# The exact prompt we used to prove it KNEW the fact in Step G
clean_prompt = "The former BBC Radio 1 DJ who stepped down from Capital Xtra after allegations emerged is Tim "

# Tokenize and move to GPU
inputs = tokenizer(clean_prompt, return_tensors="pt").to("cuda")

print("Generating response...\n")
print("-" * 40)

# Generate with mixed precision
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    out = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.0 # Greedy decoding so we see its most confident answer
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))
print("-" * 40)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Phase 3: Live Verification Test ---
Generating response...

----------------------------------------
The former BBC Radio 1 DJ who stepped down from Capital Xtra after allegations emerged is Tim  provin rond provin0 rond rond rond rond rond rond rond rond rond0000000000000000000000000000000000000
----------------------------------------


In [7]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# Cell 7: Custom Retain DataLoader

retain_df = pd.read_csv("/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/retain_set.csv")
# Extract the text column into a python list (change 'text' to your actual column name)
retain_texts = retain_df['text'].tolist()

print(f"✅ Loaded {len(retain_texts)} retain records from Drive.")

# 2. Build the PyTorch Dataset
class MUSE_RetainDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.texts = texts
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        # Tokenize the text on the fly
        encodings = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        # Squeeze removes the batch dimension added by tokenizer so DataLoader can batch it properly
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze()
        }

# 3. Create the DataLoader
retain_dataset = MUSE_RetainDataset(retain_texts, tokenizer)

# We use batch_size=1 to match our single forget prompt, keeping GPU memory low
retain_dataloader = DataLoader(retain_dataset, batch_size=1, shuffle=True)
print("✅ Retain DataLoader ready.")

✅ Loaded 3555 retain records from Drive.
✅ Retain DataLoader ready.


In [8]:
# Cell 8: Step J - Utility-Preserving Loss Architecture
class UtilityPreservingUnlearningLoss(nn.Module):
    def __init__(self, grid_margin=0.133, lambda_reg=50.0, alpha_retain=1.0):
        super().__init__()
        self.grid_margin = grid_margin
        self.lambda_reg = lambda_reg
        self.alpha_retain = alpha_retain # How strongly we enforce grammar repair

    def forward(self, forget_logits, forget_labels, retain_logits, retain_labels, current_weights, original_weights):
        # 1. FORGET LOSS (Gradient Ascent - Notice the negative sign)
        shift_f_logits = forget_logits[..., :-1, :].contiguous().float()
        shift_f_labels = forget_labels[..., 1:].contiguous()
        f_ce_loss = F.cross_entropy(shift_f_logits.view(-1, shift_f_logits.size(-1)), shift_f_labels.view(-1))
        forget_loss = -torch.clamp(f_ce_loss, max=50.0)

        # 2. RETAIN LOSS (Gradient Descent - Notice there is NO negative sign)
        # We want the model to get these words right, so we minimize this standard loss
        shift_r_logits = retain_logits[..., :-1, :].contiguous().float()
        shift_r_labels = retain_labels[..., 1:].contiguous()
        # We use ignore_index to skip padded tokens in the retain set
        retain_loss = F.cross_entropy(shift_r_logits.view(-1, shift_r_logits.size(-1)), shift_r_labels.view(-1), ignore_index=tokenizer.pad_token_id)

        # 3. GRID PENALTY (Bucket Jumping)
        weight_diff = torch.abs(current_weights - original_weights)
        grid_penalty = torch.relu(self.grid_margin - weight_diff).mean()

        # 4. TOTAL COMBINED LOSS
        total_loss = forget_loss + (self.alpha_retain * retain_loss) + (self.lambda_reg * grid_penalty)

        return total_loss, forget_loss, retain_loss, grid_penalty

criterion = UtilityPreservingUnlearningLoss()
print("✅ Multi-Objective Loss Function Initialized.")

✅ Multi-Objective Loss Function Initialized.


In [9]:
import math
import copy

# Cell 9: The Final Stage 3 Training Loop
scaler = torch.amp.GradScaler('cuda')

# The Sensitive Prompt
forget_text = "The former BBC Radio 1 DJ who stepped down from Capital Xtra after allegations emerged is Tim Westwood"
forget_inputs = tokenizer(forget_text, return_tensors="pt").to("cuda")
forget_labels = forget_inputs["input_ids"].clone()

# Create an iterator for the retain dataloader
retain_iter = iter(retain_dataloader)

epochs = 50
print("--- Starting Stage 3: Utility-Preserving Unlearning ---")

model.train()
best_grid_penalty = float('inf')
best_weights = None
last_good_epoch = 0

for epoch in range(epochs):
    optimizer.zero_grad()

    # Grab the next batch of safe, normal grammar data
    try:
        retain_batch = next(retain_iter)
    except StopIteration:
        # If we run out of retain data, restart the iterator
        retain_iter = iter(retain_dataloader)
        retain_batch = next(retain_iter)

    retain_input_ids = retain_batch['input_ids'].to("cuda")
    retain_attention = retain_batch['attention_mask'].to("cuda")
    retain_labels = retain_input_ids.clone()

    with torch.autocast("cuda", dtype=torch.float16):
        # Forward pass 1: The sensitive data
        forget_outputs = model(**forget_inputs)

        # Forward pass 2: The healthy grammar data
        retain_outputs = model(input_ids=retain_input_ids, attention_mask=retain_attention)

        current_layer30_weights = get_flat_weights(target_layer)

        # Calculate the 3-part multi-objective loss
        loss, f_loss, r_loss, g_penalty = criterion(
            forget_outputs.logits,
            forget_labels,
            retain_outputs.logits,
            retain_labels,
            current_layer30_weights,
            original_layer30_weights
        )

    if math.isnan(loss.item()):
        print(f"\n⚠️ WARNING: FP16 Cascade detected at Epoch {epoch+1}! Triggering emergency stop.")
        break

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()

    if g_penalty.item() < best_grid_penalty:
        best_grid_penalty = g_penalty.item()
        best_weights = copy.deepcopy(target_layer.state_dict())
        last_good_epoch = epoch + 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d} | Total: {loss.item():.2f} | Forget: {f_loss.item():.2f} | Retain: {r_loss.item():.2f} | Penalty: {g_penalty.item():.4f}")

if best_weights is not None:
    target_layer.load_state_dict(best_weights)
    print(f"\n✅ Restored model to safely checkpointed weights from Epoch {last_good_epoch}.")

--- Starting Stage 3: Utility-Preserving Unlearning ---
Epoch 01 | Total: -14.41 | Forget: -50.00 | Retain: 30.03 | Penalty: 0.1112
Epoch 05 | Total: -35.17 | Forget: -50.00 | Retain: 9.29 | Penalty: 0.1108
Epoch 10 | Total: -36.75 | Forget: -50.00 | Retain: 7.72 | Penalty: 0.1106
Epoch 15 | Total: -37.58 | Forget: -50.00 | Retain: 6.91 | Penalty: 0.1102
Epoch 20 | Total: -36.61 | Forget: -50.00 | Retain: 7.91 | Penalty: 0.1097
Epoch 25 | Total: -38.12 | Forget: -50.00 | Retain: 6.42 | Penalty: 0.1093
Epoch 30 | Total: -39.86 | Forget: -50.00 | Retain: 4.71 | Penalty: 0.1087
Epoch 35 | Total: -37.80 | Forget: -50.00 | Retain: 6.80 | Penalty: 0.1081
Epoch 40 | Total: -39.40 | Forget: -50.00 | Retain: 5.22 | Penalty: 0.1076
Epoch 45 | Total: -39.97 | Forget: -50.00 | Retain: 4.67 | Penalty: 0.1071
Epoch 50 | Total: -41.06 | Forget: -50.00 | Retain: 3.60 | Penalty: 0.1067

✅ Restored model to safely checkpointed weights from Epoch 50.


In [10]:
# Cell 10: Final Verification of Utility-Preserving Unlearning

print("\n--- Phase 3: Final Live Verification Test ---")
model.eval()

# The exact prompt used to trigger the "brain damage" earlier
clean_prompt = "The former BBC Radio 1 DJ who stepped down from Capital Xtra after allegations emerged is Tim "

inputs = tokenizer(clean_prompt, return_tensors="pt").to("cuda")

print("Generating response...\n")
print("-" * 40)

# Generate with mixed precision
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    out = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False # Silences the Hugging Face warning and uses greedy decoding
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))
print("-" * 40)


--- Phase 3: Final Live Verification Test ---
Generating response...

----------------------------------------
The former BBC Radio 1 DJ who stepped down from Capital Xtra after allegations emerged is Tim  Johnson

















































----------------------------------------
